# Metasyn Tutorial: Time-Series Dependencies and Logical Constraints

In this tutorial, we explore how to model time-series data with start and end dates, and how to add column dependency relationships and constraints to synthetic data using the implemented dunder methods.


### 0. Install and import

First, let's install metasyn if you haven't done so already.

In [1]:
# %pip install metasyn

Now let's import the packages we need.

In [2]:
import polars as pl

from metasyn import demo_data
from metasyn.builder import MetaFrameBuilder
from metasyn.distribution import DiscreteTruncatedNormalDistribution
from metasyn.distribution.base import (
    ColumnReference,
    IfThenElse,
)

### 1. Synthesizing time-series data

A common challenge with time-series datasets is that columns can be related to each other. For example, in a hospital admissions dataset the `discharge_date` should always come *after* the `admission_date`. By default, metasyn treats each column independently, so this constraint is not preserved.

Let's load our hospital admissions dataset to see this in action.

In [3]:
df = demo_data("hospital_admissions")

df.head()

Patient_id,Admission_date,Discharge_date,Length_of_stay_days,Age,Height_cm,Weight_kg,Sex
i64,date,date,i64,i64,i64,i64,cat
1,2023-01-04,2023-01-06,2,44,164,67,"""F"""
2,2023-01-08,2023-01-18,10,78,154,57,"""F"""
3,2023-01-08,2023-01-19,11,18,167,77,"""M"""
4,2023-01-08,2023-01-24,16,83,177,95,"""F"""
5,2023-01-15,2023-01-21,6,9,138,25,"""M"""


If we fit and synthesize without any additional instructions, the dates are generated independently. Let's check how often this produces inconsistent results.

In [4]:
builder = MetaFrameBuilder()
builder.add_dataframe(df, None)

synth = builder.fit().synthesize()

n_inconsistent = (synth["Discharge_date"] < synth["Admission_date"]).sum()
print(f"\nRows where Discharge_date < Admission_date: {n_inconsistent} / {len(synth)}")

synth.head()

  0%|          | 0/8 [00:00<?, ?it/s]

  Patient_id: 100%|██████████| 8/8 [00:00<00:00, 607.95variables/s]


Rows where Discharge_date < Admission_date: 51 / 100


Patient_id,Admission_date,Discharge_date,Length_of_stay_days,Age,Height_cm,Weight_kg,Sex
i64,date,date,i64,i64,i64,i64,cat
1,2024-03-01,2023-10-23,14,79,171,88,"""M"""
2,2023-01-11,2024-01-02,9,2,178,67,"""M"""
3,2023-11-02,2023-09-20,7,80,163,22,"""F"""
4,2023-04-05,2024-04-11,4,56,139,94,"""F"""
5,2023-01-19,2023-04-18,16,9,133,77,"""F"""


As we can see, there are some inconsistent rows. To fix this, we can add a hidden `duration_days` column that holds the difference in days between `Discharge_date` and `Admission_date`. Metasyn fits a distribution to the new column, which can then be used to compute `Discharge_date = Admission_date + duration_days`. Setting `hidden=True` ensures the helper column does **not** appear in the final output.

Note that `ColumnReference` refers to the values in a column at synthesis time, and can be used to build expressions between columns.

In [5]:
builder = MetaFrameBuilder()
builder.add_dataframe(df, None)

# Add a column and mark as hidden
builder.add_column("duration_days", hidden=True)

# Create a series for the difference between discharge_date and admission_date, and use it to derive discharge_date
builder["duration_days"].series = ColumnReference("Discharge_date") - ColumnReference("Admission_date")
builder["Discharge_date"].distribution = ColumnReference("Admission_date") + ColumnReference("duration_days")

synth = builder.fit().synthesize()

n_inconsistent = (synth["Discharge_date"] < synth["Admission_date"]).sum()
print(f"\nRows where Discharge_date < Admission_date: {n_inconsistent} / {len(synth)}")

synth

  Patient_id: 100%|██████████| 9/9 [00:00<00:00, 455.56variables/s]


Rows where Discharge_date < Admission_date: 0 / 100


Patient_id,Admission_date,Discharge_date,Length_of_stay_days,Age,Height_cm,Weight_kg,Sex
i64,date,date,i64,i64,i64,i64,cat
1,2024-01-13,2024-01-19,4,46,155,60,"""F"""
2,2024-08-10,2024-08-16,12,17,193,27,"""F"""
3,2023-08-21,2023-09-04,11,77,168,35,"""F"""
4,2024-05-14,2024-05-17,17,59,183,17,"""M"""
5,2023-06-10,2023-06-27,15,63,180,36,"""M"""
…,…,…,…,…,…,…,…
96,2023-07-04,2023-07-09,5,75,187,40,"""F"""
97,2023-03-24,2023-04-04,6,55,195,69,"""F"""
98,2023-07-17,2023-07-31,7,72,133,-15,"""M"""


No more inconsistent rows.

This dataframe already contains a duration column, `Length_of_stay_days`, so we do not need to create a hidden helper column for this example. Instead, we convert `Length_of_stay_days` from an integer column to a Polars duration column, so metasyn recognizes it as a duration.

In [6]:
# Check the Polars documentation for the right way to convert your column to a duration.
df_duration = df.with_columns(
    pl.duration(days=pl.col("Length_of_stay_days")).alias("Length_of_stay_days")
)

df_duration.head()

Patient_id,Admission_date,Discharge_date,Length_of_stay_days,Age,Height_cm,Weight_kg,Sex
i64,date,date,duration[μs],i64,i64,i64,cat
1,2023-01-04,2023-01-06,2d,44,164,67,"""F"""
2,2023-01-08,2023-01-18,10d,78,154,57,"""F"""
3,2023-01-08,2023-01-19,11d,18,167,77,"""M"""
4,2023-01-08,2023-01-24,16d,83,177,95,"""F"""
5,2023-01-15,2023-01-21,6d,9,138,25,"""M"""


Because metasyn supports duration columns, we can define `Discharge_date` as the sum of `Admission_date` and `Length_of_stay_days`.

In [15]:
builder = MetaFrameBuilder()
builder.add_dataframe(df_duration, None)

builder["Discharge_date"].distribution = ColumnReference("Admission_date") + ColumnReference("Length_of_stay_days")

synth_df = builder.fit().synthesize()

synth_df

  Patient_id: 100%|██████████| 8/8 [00:00<00:00, 445.42variables/s]


Patient_id,Admission_date,Discharge_date,Length_of_stay_days,Age,Height_cm,Weight_kg,Sex
i64,date,date,duration[μs],i64,i64,i64,cat
1,2023-04-08,2023-04-28,20d,31,173,84,"""F"""
2,2024-01-25,2024-02-01,7d,57,143,117,"""M"""
3,2024-09-04,2024-09-21,17d,35,171,59,"""M"""
4,2024-10-04,2024-10-19,15d,6,186,92,"""F"""
5,2024-01-15,2024-02-03,19d,7,178,68,"""M"""
…,…,…,…,…,…,…,…
96,2023-11-18,2023-11-30,12d,5,176,55,"""F"""
97,2024-07-19,2024-07-27,8d,21,166,94,"""F"""
98,2023-01-07,2023-01-17,10d,85,193,68,"""M"""


The synthesized `Discharge_date` values now align with the corresponding `Admission_date` and `Length_of_stay_days` values.

In [16]:
# Convert the duration column back to integer type that represents a number of days like the original column.
# Check the Polars documentation for the right way to convert your column back to integer type.
synth_df = synth_df.with_columns(
    pl.col("Length_of_stay_days").dt.total_days().alias("Length_of_stay_days")
)

synth_df

Patient_id,Admission_date,Discharge_date,Length_of_stay_days,Age,Height_cm,Weight_kg,Sex
i64,date,date,i64,i64,i64,i64,cat
1,2023-04-08,2023-04-28,20,31,173,84,"""F"""
2,2024-01-25,2024-02-01,7,57,143,117,"""M"""
3,2024-09-04,2024-09-21,17,35,171,59,"""M"""
4,2024-10-04,2024-10-19,15,6,186,92,"""F"""
5,2024-01-15,2024-02-03,19,7,178,68,"""M"""
…,…,…,…,…,…,…,…
96,2023-11-18,2023-11-30,12,5,176,55,"""F"""
97,2024-07-19,2024-07-27,8,21,166,94,"""F"""
98,2023-01-07,2023-01-17,10,85,193,68,"""M"""


### 2. Logical expressions and conditions

If a boolean column is fully determined by another column, you can encode that dependency directly. For example, if the dataset contains an `Adult` column, you can define it as `builder["Adult"].distribution = ColumnReference("Age") > 18`. This keeps synthesized data internally consistent without post-processing.


The comparison and logical operators let you derive boolean columns from expressions. The table below lists the implemented operators and example usage. For readability, the examples show column names directly, but in practice, reference a column with `ColumnReference()`, for example `ColumnReference("Age")`.

| Dunder | Operator | Example |
|---|---|---|
| `__add__` | `a + b` | `Age + 10` |
| `__sub__` | `a - b` | `Age - 10` |
| `__mul__` | `a * b` | `Weight_kg * 2` |
| `__truediv__` | `a / b` | `Weight_kg / 2` |
| `__pow__` | `a ** b` | `Weight_kg ** 2` |
| `__neg__` | `-a` | `-Weight_kg` |
| `__invert__` | `not a` | `not Adult` |
| `__and__` | `a & b` | `Adult & (Sex == "male")` |
| `__or__` | `a \| b` | `Adult \| Child` |
| `__eq__` | `a == b` | `Sex == "female"` |
| `__ne__` | `a != b` | `Sex != "male"` |
| `__lt__` | `a < b` | `Age < 18` |
| `__gt__` | `a > b` | `Age > 17` |
| `__le__` | `a <= b` | `Age <= 17` |
| `__ge__` | `a >= b` | `Age >= 18` |



In [8]:
# Example without relations or constraints

# Create example data with boolean "Adult" column
df = df.with_columns(
    (pl.col("Age") > 18).alias("Adult"),
)

builder = MetaFrameBuilder()
builder.add_dataframe(df, None)

synth = builder.fit().synthesize()

n_mismatches = synth.filter(
    (pl.col("Age") > 18) & ~pl.col("Adult")
).height

print(f"\nRows where age > 18 but Adult is false: {n_mismatches} / {len(synth)}")

synth[["Age", "Adult"]]

  Patient_id: 100%|██████████| 9/9 [00:00<00:00, 493.28variables/s]


Rows where age > 18 but Adult is false: 22 / 100


Age,Adult
i64,bool
45,true
18,true
15,false
80,true
35,false
…,…
75,true
39,false
47,false


Let's fix the inconsistency.

In [9]:
# Add relations or constraints
builder["Adult"].distribution = ColumnReference("Age") > 18

synth = builder.fit().synthesize()

n_mismatches = synth.filter(
    (pl.col("Age") > 18) & ~pl.col("Adult")
).height

print(f"\nRows where age > 18 but Adult is false: {n_mismatches} / {len(synth)}")

synth[["Age", "Adult"]]

 44%|████▍     | 4/9 [00:00<00:00,  7.46it/s]


KeyboardInterrupt: 

#### IfThenElse example

Sometimes the distribution of a column depends on the value of another column. For example, `Height_cm` tends to differ between male and female patients. With `IfThenElse`, you can specify a different distribution or value for each group.

In [ ]:
# Males: TruncatedNormal centred at 180 cm; females: centred at 170 cm
builder["Height_cm"].distribution = IfThenElse(
    ColumnReference("Sex") == "M",
    DiscreteTruncatedNormalDistribution(lower=160, upper=200, mean=180, sd=10),
    DiscreteTruncatedNormalDistribution(lower=150, upper=190, mean=170, sd=10),
)

synth = builder.fit().synthesize()
synth[["Sex", "Height_cm"]].head(10)

  Patient_id: 100%|██████████| 9/9 [00:00<00:00, 468.38variables/s]


Sex,Height_cm
cat,i64
"""M""",173
"""F""",166
"""F""",172
"""F""",185
"""F""",157
"""M""",178
"""F""",179
"""M""",174
"""F""",185


Actually, this dataset includes more complex dependencies: `Height_cm` and `Weight_kg` are strongly associated with age and body size.
For example, a 6-year-old is very unlikely to be 200 cm tall or weigh 100 kg.

To make synthetic data more realistic, we can divide age and height into broad, plausible categories:

- **0-2 years:** height 50-95 cm
- **3-12 years:** height 90-165 cm
- **13-17 years:** height 145-195 cm
- **18+ years:** height 145-210 cm
- **Up to 95 cm:** weight 3-16 kg
- **96-165 cm:** weight 14-65 kg
- **166-185 cm:** weight 45-100 kg
- **Above 185 cm:** weight 60-140 kg

These ranges are useful defaults for synthetic data generation, but edge-case combinations can still appear.

In the example below, we combine logical and conditional operators to generate `Height_cm` from `Age`, and then generate `Weight_kg` from the synthesized `Height_cm`.

In [ ]:
builder = MetaFrameBuilder()
builder.add_dataframe(df, None)

builder["Height_cm"].distribution = IfThenElse(
    ColumnReference("Age") <= 2,
    DiscreteTruncatedNormalDistribution(lower=50, upper=95, mean=80, sd=12),
    IfThenElse(
        ColumnReference("Age") <= 12,
        DiscreteTruncatedNormalDistribution(lower=90, upper=165, mean=130, sd=18),
        IfThenElse(
            ColumnReference("Age") <= 17,
            DiscreteTruncatedNormalDistribution(lower=145, upper=195, mean=170, sd=12),
            DiscreteTruncatedNormalDistribution(lower=145, upper=210, mean=172, sd=11),
        ),
    ),
)

builder["Weight_kg"].distribution = IfThenElse(
    ColumnReference("Height_cm") <= 95,
    DiscreteTruncatedNormalDistribution(lower=3, upper=16, mean=11, sd=3),
    IfThenElse(
        ColumnReference("Height_cm") <= 165,
        DiscreteTruncatedNormalDistribution(lower=14, upper=65, mean=38, sd=12),
        IfThenElse(
            ColumnReference("Height_cm") <= 185,
            DiscreteTruncatedNormalDistribution(lower=45, upper=100, mean=72, sd=13),
            DiscreteTruncatedNormalDistribution(lower=60, upper=140, mean=90, sd=15),
        ),
    ),
)

synth = builder.fit().synthesize()

synth

  Patient_id: 100%|██████████| 9/9 [00:00<00:00, 380.93variables/s]


Patient_id,Admission_date,Discharge_date,Length_of_stay_days,Age,Height_cm,Weight_kg,Sex,Adult
i64,date,date,i64,i64,i64,i64,cat,bool
1,2023-11-08,2023-10-05,10,53,170,82,"""M""",true
2,2024-02-24,2024-08-20,11,15,163,54,"""M""",true
3,2023-04-03,2023-09-05,18,83,180,81,"""M""",true
4,2023-05-23,2023-07-21,7,43,171,71,"""F""",true
5,2023-11-02,2023-09-06,13,28,158,72,"""F""",false
…,…,…,…,…,…,…,…,…
96,2024-03-28,2023-07-16,3,18,180,74,"""F""",true
97,2024-03-03,2023-08-08,8,17,157,42,"""F""",true
98,2024-09-17,2024-01-21,5,58,189,93,"""M""",true


As you can see, height follows the age groups from the original data, and weight follows the generated height groups.

### 3. Important note

When defining relationships between columns, it is usually safer to **preserve the broad structure of the data** than to reproduce every numerical detail exactly. Metasyn can help reduce disclosure risk through its modelling choices and constraints, but it cannot prevent users from encoding **sensitive information indirectly through overly specific rules**. Aim for relationships that are realistic enough for the intended use case, while still remaining approximate.